# Notebook 4: Two-Zone Model with UltraNest

**Object:** MG3 J163554+3629 (z = 3.648)
**Model:** Two-zone (compact EC+SSC + extended SSC) × EBL (Franceschini 2008)

Loads the two-zone prefit model from Notebook 3 and runs UltraNest nested sampling.
Free parameters span both zones. Shared parameters are kept fixed (linked between zones).
The output log Z can be compared to the one-zone log Z (Notebook 2) for Bayesian model comparison.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import dill, corner, ultranest, ultranest.stepsampler
%matplotlib inline

from jetset.data_loader import ObsData
from jetset.model_manager import FitModel

two_zone_model = FitModel.load_model('../../results/models/twozone_model.pkl')
sed_data       = ObsData.load('../../results/models/sed_data.pkl')

print('Two-zone model loaded.')
two_zone_model.show_model()

ModuleNotFoundError: No module named 'ultranest'

---
## Setup: freeze all, select free parameters

Free parameters for Zone 1 (compact):
- B, BulkFactor, N, M_BH, L_Disk, T_DT (6 params, same as 1-zone UltraNest)

Free parameters for Zone 2 (extended):
- B_z2, N_z2 (2 extra params — R_H_z2 is tied to R_DT of Zone 1)

Shared params (z, theta, Gamma, L_Disk, M_BH, etc.) are inherited from Zone 1 linking.

In [ ]:
# Freeze all parameters
for par in two_zone_model.jet_compact.parameters.par_array:
    par.frozen = True
for par in two_zone_model.jet_extended.parameters.par_array:
    par.frozen = True

# Set frozen params to best-fit Minuit values
p1 = two_zone_model.jet_compact.parameters
p1.p.val           = 1.590
p1.p_1.val         = 3.066
p1.gmin.val        = 2.101
p1.gmax.val        = 1.653e4
p1.gamma_break.val = 381.0
p1.accr_eff.val    = 0.083

# Zone 2 electron shape frozen to Zone 1 values
p2 = two_zone_model.jet_extended.parameters
p2.p.val           = p1.p.val
p2.p_1.val         = p1.p_1.val
p2.gamma_break.val = p1.gamma_break.val
p2.gmin.val        = p1.gmin.val
p2.gmax.val        = p1.gmax.val

# Free parameters: Zone1 (6) + Zone2 B and N (2) = 8 total
free_z1 = ['B', 'BulkFactor', 'N', 'M_BH', 'L_Disk', 'T_DT']
free_z2 = ['B', 'N']  # Zone 2 independent B and N

for name in free_z1:
    two_zone_model.jet_compact.parameters.get_par_by_name(name).frozen = False
for name in free_z2:
    two_zone_model.jet_extended.parameters.get_par_by_name(name).frozen = False

# All free parameter names for UltraNest (prefix zone)
free_params = ['z1_B','z1_BulkFactor','z1_N','z1_M_BH','z1_L_Disk','z1_T_DT',
               'z2_B','z2_N']

# Prior bounds: Zone 1 same as notebook 2; Zone 2 independent
prior_mins = np.array([0.020, 10.0,  500.,  5e8,  1e46, 300.,  1e-4, 1e-2])
prior_maxs = np.array([0.150, 25.0, 5000.,  5e9,  4e46, 1200., 0.02, 10.0])

print(f'Free parameters ({len(free_params)}): {free_params}')

In [ ]:
# Prepare data arrays
nu_obs   = sed_data.table['nu_data'].data.astype(float)
flux_obs = sed_data.table['nuFnu_data'].data.astype(float)
flux_err = sed_data.table['dnuFnu_data'].data.astype(float)
is_ul    = sed_data.table['UL'].data.astype(bool)

def get_composite_flux(nu_target):
    two_zone_model.eval()
    nu_grid   = two_zone_model.SED.nu.value
    flux_grid = two_zone_model.SED.nuFnu.value
    good = np.isfinite(flux_grid) & np.isfinite(nu_grid) & (nu_grid > 0)
    nu_g, fl_g = nu_grid[good], flux_grid[good]
    idx = np.argsort(nu_g)
    return np.interp(nu_target, nu_g[idx], fl_g[idx], left=0., right=0.)

def prior_transform(cube):
    return prior_mins + cube * (prior_maxs - prior_mins)

def log_likelihood(params):
    try:
        z1_B, z1_G, z1_N, z1_MBH, z1_LD, z1_TDT, z2_B, z2_N = params
        two_zone_model.jet_compact.set_par('B',           val=z1_B)
        two_zone_model.jet_compact.set_par('BulkFactor',  val=z1_G)
        two_zone_model.jet_compact.set_par('N',           val=z1_N)
        two_zone_model.jet_compact.set_par('M_BH',        val=z1_MBH)
        two_zone_model.jet_compact.set_par('L_Disk',      val=z1_LD)
        two_zone_model.jet_compact.set_par('T_DT',        val=z1_TDT)
        two_zone_model.jet_extended.set_par('B',          val=z2_B)
        two_zone_model.jet_extended.set_par('N',          val=z2_N)
        model_flux = get_composite_flux(nu_obs)
        if np.any(~np.isfinite(model_flux)):
            return -1e100
        chi2 = 0.0
        for i in range(len(nu_obs)):
            if flux_err[i] <= 0: continue
            if is_ul[i]:
                if model_flux[i] > flux_obs[i]:
                    chi2 += ((model_flux[i] - flux_obs[i]) / flux_err[i])**2
            else:
                if model_flux[i] > 0:
                    chi2 += ((model_flux[i] - flux_obs[i]) / flux_err[i])**2
        return -0.5 * chi2 if np.isfinite(chi2) else -1e100
    except Exception:
        return -1e100

# Sanity check
best_test = np.array([0.05, 15.0, 1000., 1.9e9, 1.58e46, 600., 0.005, 0.1])
ll_test = log_likelihood(best_test)
print(f'logL sanity check: {ll_test:.4f}')
assert ll_test > -1e99, 'Likelihood broken.'

In [ ]:
# Run UltraNest
sampler = ultranest.ReactiveNestedSampler(
    param_names = free_params,
    loglike     = log_likelihood,
    transform   = prior_transform,
    log_dir     = '../../results/ultranest_output_2zone',
    resume      = True
)
sampler.stepsampler = ultranest.stepsampler.SliceSampler(
    nsteps             = 2 * len(free_params),
    generate_direction = ultranest.stepsampler.generate_mixture_random_direction
)

print(f'Running UltraNest: {len(free_params)} parameters, 400 live points...')
results = sampler.run(
    min_num_live_points = 400,
    max_iters           = 150000,
    show_status         = True,
    viz_callback        = False
)
sampler.print_results()

In [ ]:
# Results summary
samples  = results['samples']
post_med = np.median(samples, axis=0)
post_lo  = np.percentile(samples, 16, axis=0)
post_hi  = np.percentile(samples, 84, axis=0)

print('\n=== Two-zone UltraNest posterior ===')
for name, med, lo, hi in zip(free_params, post_med, post_lo, post_hi):
    print(f'  {name:20s}: {med:.4e}  +{hi-med:.4e}  -{med-lo:.4e}')
print(f'\nlog Z (2-zone) = {results["logz"]:.3f} +/- {results["logzerr"]:.3f}')
print('Compare to log Z (1-zone) from Notebook 2 to get Bayes factor.')

# Save
np.save('../../results/models/ultranest_samples_2zone.npy', samples)
np.save('../../results/models/ultranest_results_2zone.npy', results, allow_pickle=True)

# Set model to posterior median and plot
two_zone_model.jet_compact.set_par('B',          val=post_med[0])
two_zone_model.jet_compact.set_par('BulkFactor', val=post_med[1])
two_zone_model.jet_compact.set_par('N',          val=post_med[2])
two_zone_model.jet_compact.set_par('M_BH',       val=post_med[3])
two_zone_model.jet_compact.set_par('L_Disk',     val=post_med[4])
two_zone_model.jet_compact.set_par('T_DT',       val=post_med[5])
two_zone_model.jet_extended.set_par('B',         val=post_med[6])
two_zone_model.jet_extended.set_par('N',         val=post_med[7])
two_zone_model.eval()

p = two_zone_model.plot_model(sed_data=sed_data)
p.add_residual_plot(two_zone_model, sed_data)
p.setlim(y_min=1E-16, x_min=1E8)
plt.suptitle('MG3 J163554+3629 — Two-zone UltraNest posterior median')
plt.savefig('../../results/plots/ultranest_bestfit_sed_2zone.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Posterior envelope plot for 2-zone model
n_draw = 150
idx_draw = np.random.choice(len(samples), size=n_draw, replace=False)
flux_matrix = []
nu_ref = None

for idx in idx_draw:
    s = samples[idx]
    two_zone_model.jet_compact.set_par('B',          val=s[0])
    two_zone_model.jet_compact.set_par('BulkFactor', val=s[1])
    two_zone_model.jet_compact.set_par('N',          val=s[2])
    two_zone_model.jet_compact.set_par('M_BH',       val=s[3])
    two_zone_model.jet_compact.set_par('L_Disk',     val=s[4])
    two_zone_model.jet_compact.set_par('T_DT',       val=s[5])
    two_zone_model.jet_extended.set_par('B',         val=s[6])
    two_zone_model.jet_extended.set_par('N',         val=s[7])
    two_zone_model.eval()
    nu_g   = two_zone_model.SED.nu.value
    flux_g = two_zone_model.SED.nuFnu.value
    good   = np.isfinite(flux_g) & np.isfinite(nu_g) & (nu_g > 0) & (flux_g > 0)
    if nu_ref is None: nu_ref = nu_g[good]
    flux_matrix.append(np.interp(nu_ref, nu_g[good], flux_g[good], left=0., right=0.))

flux_matrix = np.array(flux_matrix)
fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(nu_ref, np.percentile(flux_matrix,16,axis=0),
                np.percentile(flux_matrix,84,axis=0),
                alpha=0.35, color='darkorange', label='posterior 68% band')
ax.loglog(nu_ref, np.percentile(flux_matrix,50,axis=0),
          color='darkorange', lw=2, label='median model')

mask_ul  = sed_data.table['UL'].astype(bool)
mask_det = ~mask_ul
ax.errorbar(sed_data.table['nu_data'][mask_det],
            sed_data.table['nuFnu_data'][mask_det],
            yerr=sed_data.table['dnuFnu_data'][mask_det],
            fmt='o', color='navy', ms=4, label='data', zorder=5)
ax.scatter(sed_data.table['nu_data'][mask_ul],
           sed_data.table['nuFnu_data'][mask_ul],
           marker='v', color='navy', s=30, zorder=5)

ax.set_xlim(1e8,1e27); ax.set_ylim(1e-16,1e-10)
ax.set_xlabel(r'$\nu$ (Hz)', fontsize=13)
ax.set_ylabel(r'$\nu F_\nu$ (erg cm$^{-2}$ s$^{-1}$)', fontsize=13)
ax.set_title('MG3 J163554+3629 — UltraNest posterior envelope (2-zone)', fontsize=12)
ax.legend(fontsize=11); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('../../results/plots/ultranest_envelope_2zone.png', dpi=150, bbox_inches='tight')
plt.show()